# 17.1 持续学习 (Continual Learning)

持续学习使模型在不遗忘旧知识的情况下学习新知识，是模型长期部署的核心能力。

本节涵盖：
- 灾难性遗忘问题
- 正则化方法（EWC/SI/MAS）
- 回放方法（经验回放/生成回放）
- 架构方法（渐进网络/PackNet）
- LLM持续学习
- 持续学习评估

## 1. 灾难性遗忘

**核心问题**：神经网络学习新任务时会覆盖旧任务的知识。

### 稳定性-可塑性困境
- **稳定性**：保留旧知识（过强→无法学习新知识）
- **可塑性**：学习新知识（过强→遗忘旧知识）

### 三种场景
| 场景 | 任务标识 | 旧任务数据 | 难度 |
|------|---------|-----------|------|
| Task-Incremental | 已知 | 可用 | 低 |
| Domain-Incremental | 未知 | 可用 | 中 |
| Class-Incremental | 未知 | 不可用 | 高 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

torch.manual_seed(42)

class SimpleModel(nn.Module):
    def __init__(self, d=64, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, n_classes))
    def forward(self, x):
        return self.net(x)

def train_task(model, x, y, n_steps=20, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(n_steps):
        loss = F.cross_entropy(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

x_task1 = torch.randn(32, 64)
y_task1 = torch.randint(0, 5, (32,))
x_task2 = torch.randn(32, 64)
y_task2 = torch.randint(5, 10, (32,))

model_naive = SimpleModel()
train_task(model_naive, x_task1, y_task1)
with torch.no_grad():
    task1_acc_before = (model_naive(x_task1).argmax(1) == y_task1).float().mean()
train_task(model_naive, x_task2, y_task2)
with torch.no_grad():
    task1_acc_after_naive = (model_naive(x_task1).argmax(1) == y_task1).float().mean()
    task2_acc_naive = (model_naive(x_task2).argmax(1) == y_task2).float().mean()

print('=== Catastrophic Forgetting Demo ===')
print(f'Task 1 accuracy before learning Task 2: {task1_acc_before:.2%}')
print(f'Task 1 accuracy after learning Task 2 (naive): {task1_acc_after_naive:.2%}')
print(f'Task 2 accuracy (naive): {task2_acc_naive:.2%}')
print(f'Forgetting: {task1_acc_before - task1_acc_after_naive:.2%}')
print(f'\nKey: Naive sequential training causes severe forgetting of Task 1.')

## 2. 正则化方法

### EWC (Elastic Weight Consolidation)
对重要权重施加更大的约束，重要性通过Fisher信息矩阵估计：
$$L_{EWC} = L_{task} + \frac{\lambda}{2} \sum_i F_i (\theta_i - \theta_i^*)^2$$

### SI (Synaptic Intelligence)
在线计算参数重要性，无需存储Fisher矩阵：
- 追踪每个参数对损失的贡献
- 重要性 = 累计梯度×参数变化

### MAS (Memory Aware Synapses)
基于输出敏感度计算重要性，不依赖标签：
- 重要性 = 输出对参数的梯度范数
- 适用于无标签场景

### 对比
| 方法 | 重要性计算 | 需要标签 | 计算成本 |
|------|-----------|---------|----------|
| EWC | Fisher信息 | 是 | 高 |
| SI | 在线梯度 | 是 | 低 |
| MAS | 输出敏感度 | 否 | 中 |

In [ ]:
class EWC:
    def __init__(self, model, x, y, lambda_ewc=0.5):
        self.model = model
        self.lambda_ewc = lambda_ewc
        self.params_ref = {n: p.clone() for n, p in model.named_parameters()}
        self.fisher = {n: torch.zeros_like(p) for n, p in model.named_parameters()}
        model.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        for n, p in model.named_parameters():
            if p.grad is not None:
                self.fisher[n] = p.grad.data.clone() ** 2

    def penalty(self, model):
        loss = 0
        for n, p in model.named_parameters():
            loss += (self.fisher[n] * (p - self.params_ref[n]) ** 2).sum()
        return self.lambda_ewc * loss

class SI:
    def __init__(self, model, c=0.1):
        self.model = model
        self.c = c
        self.omega = {n: torch.zeros_like(p) for n, p in model.named_parameters()}
        self.prev_params = {n: p.clone() for n, p in model.named_parameters()}
        self.W = {n: torch.zeros_like(p) for n, p in model.named_parameters()}

    def update_omega(self):
        for n, p in self.model.named_parameters():
            delta = p.detach() - self.prev_params[n]
            self.omega[n] += self.W[n] / (delta ** 2 + self.c)
            self.prev_params[n] = p.clone()
            self.W[n] = torch.zeros_like(p)

    def track_gradient(self):
        for n, p in self.model.named_parameters():
            if p.grad is not None:
                self.W[n] += -p.grad * p.detach()

    def penalty(self, model):
        loss = 0
        for n, p in model.named_parameters():
            loss += (self.omega[n] * (p - self.prev_params[n]) ** 2).sum()
        return loss

model_ewc = SimpleModel()
train_task(model_ewc, x_task1, y_task1)
ewc = EWC(model_ewc, x_task1, y_task1, lambda_ewc=5.0)

optimizer = torch.optim.Adam(model_ewc.parameters(), lr=0.01)
for step in range(20):
    loss = F.cross_entropy(model_ewc(x_task2), y_task2) + ewc.penalty(model_ewc)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    task1_acc_ewc = (model_ewc(x_task1).argmax(1) == y_task1).float().mean()
    task2_acc_ewc = (model_ewc(x_task2).argmax(1) == y_task2).float().mean()

print('=== Regularization Methods ===')
print(f'Naive:  Task1={task1_acc_after_naive:.2%}, Task2={task2_acc_naive:.2%}')
print(f'EWC:    Task1={task1_acc_ewc:.2%}, Task2={task2_acc_ewc:.2%}')
print(f'EWC preserves Task1 better (less forgetting) at slight Task2 cost.')

model_si = SimpleModel()
train_task(model_si, x_task1, y_task1)
si = SI(model_si, c=0.1)
optimizer = torch.optim.Adam(model_si.parameters(), lr=0.01)
for step in range(20):
    loss = F.cross_entropy(model_si(x_task2), y_task2)
    optimizer.zero_grad()
    loss.backward()
    si.track_gradient()
    optimizer.step()
si.update_omega()

with torch.no_grad():
    task1_acc_si = (model_si(x_task1).argmax(1) == y_task1).float().mean()
    task2_acc_si = (model_si(x_task2).argmax(1) == y_task2).float().mean()
print(f'SI:     Task1={task1_acc_si:.2%}, Task2={task2_acc_si:.2%}')
print(f'\nKey: EWC uses Fisher information, SI uses online gradient tracking.')
print(f'Both penalize changes to important parameters, but importance is computed differently.')

## 3. 回放方法

### 经验回放 (Experience Replay)
保留少量旧任务数据，与新任务数据混合训练。

### 生成回放 (Generative Replay)
用生成模型生成伪旧数据，无需存储真实数据。

### 梯度回放 (Gradient Episodic Memory, GEM)
约束新任务的梯度不与旧任务梯度冲突：
$$\min_g \|g - g_{new}\|^2 \quad s.t. \quad g^T g_{old}^k \geq 0, \forall k$$

### 对比
| 方法 | 存储需求 | 隐私 | 效果 |
|------|---------|------|------|
| 经验回放 | 小缓冲区 | 低 | 好 |
| 生成回放 | 生成模型 | 高 | 中 |
| GEM | 小缓冲区 | 低 | 最好 |
| A-GEM | 小缓冲区 | 低 | 好+快 |

In [ ]:
class ExperienceReplay:
    def __init__(self, buffer_size=100):
        self.buffer_size = buffer_size
        self.buffer_x = []
        self.buffer_y = []

    def add(self, x, y):
        for i in range(x.size(0)):
            if len(self.buffer_x) >= self.buffer_size:
                idx = torch.randint(0, len(self.buffer_x), (1,)).item()
                self.buffer_x[idx] = x[i]
                self.buffer_y[idx] = y[i]
            else:
                self.buffer_x.append(x[i])
                self.buffer_y.append(y[i])

    def sample(self, n):
        n = min(n, len(self.buffer_x))
        if n == 0:
            return None, None
        indices = torch.randint(0, len(self.buffer_x), (n,))
        return torch.stack([self.buffer_x[i] for i in indices]), torch.tensor([self.buffer_y[i] for i in indices])

class GEM:
    def __init__(self, model, memory_size=50):
        self.model = model
        self.memory_size = memory_size
        self.memory_x = {}
        self.memory_y = {}

    def store_memory(self, task_id, x, y):
        n = min(self.memory_size, x.size(0))
        self.memory_x[task_id] = x[:n]
        self.memory_y[task_id] = y[:n]

    def project_gradient(self, task_id, x_new, y_new):
        self.model.zero_grad()
        loss_new = F.cross_entropy(self.model(x_new), y_new)
        grads_new = torch.autograd.grad(loss_new, self.model.parameters(), retain_graph=True)
        grads_new_flat = torch.cat([g.flatten() for g in grads_new])
        for old_id in self.memory_x:
            if old_id == task_id:
                continue
            self.model.zero_grad()
            loss_old = F.cross_entropy(self.model(self.memory_x[old_id]), self.memory_y[old_id])
            grads_old = torch.autograd.grad(loss_old, self.model.parameters())
            grads_old_flat = torch.cat([g.flatten() for g in grads_old])
            dot = torch.dot(grads_new_flat, grads_old_flat)
            if dot < 0:
                grads_new_flat = grads_new_flat - (dot / (grads_old_flat.norm() ** 2 + 1e-8)) * grads_old_flat
        return grads_new_flat

model_replay = SimpleModel()
train_task(model_replay, x_task1, y_task1)
replay = ExperienceReplay(buffer_size=50)
replay.add(x_task1, y_task1)

optimizer = torch.optim.Adam(model_replay.parameters(), lr=0.01)
for step in range(20):
    loss_new = F.cross_entropy(model_replay(x_task2), y_task2)
    x_old, y_old = replay.sample(16)
    if x_old is not None:
        loss_old = F.cross_entropy(model_replay(x_old), y_old)
        loss = 0.5 * loss_new + 0.5 * loss_old
    else:
        loss = loss_new
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    task1_acc_replay = (model_replay(x_task1).argmax(1) == y_task1).float().mean()
    task2_acc_replay = (model_replay(x_task2).argmax(1) == y_task2).float().mean()

print('=== Replay Methods ===')
print(f'Naive:  Task1={task1_acc_after_naive:.2%}, Task2={task2_acc_naive:.2%}')
print(f'EWC:    Task1={task1_acc_ewc:.2%}, Task2={task2_acc_ewc:.2%}')
print(f'Replay: Task1={task1_acc_replay:.2%}, Task2={task2_acc_replay:.2%}')
print(f'\nKey: Experience replay mixes old data with new, preventing forgetting directly.')
print(f'GEM projects gradients to avoid interference with old task gradients.')

## 4. 架构方法与LLM持续学习

### 渐进网络 (Progressive Networks)
每学一个新任务添加一列（新网络），通过横向连接利用旧知识。

### PackNet
迭代剪枝-重分配：旧任务用部分参数，新任务用剩余参数。

### LLM持续学习
LLM的持续学习面临独特挑战：
- **规模大**：全参数微调成本高
- **知识广**：遗忘影响范围大
- **解决方案**：
  - LoRA持续学习：每个任务一个LoRA适配器
  - O-LoRA：正交LoRA子空间
  - InfLoRA：无穷小扰动分析
  - 知识编辑：ROME/MEMIT精确修改
  - 检索增强：用RAG补充新知识

### 持续学习评估指标
- **平均精度**：$AA = \frac{1}{T}\sum_{i=1}^T a_{T,i}$
- **遗忘度量**：$F = \frac{1}{T-1}\sum_{i=1}^{T-1} \max_j a_{j,i} - a_{T,i}$
- **前向迁移**：新任务是否受益于旧任务
- **后向迁移**：旧任务是否受益于新任务

In [ ]:
class LoRAContinualLearning(nn.Module):
    def __init__(self, d=64, n_classes=10, rank=4):
        super().__init__()
        self.base = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.head = nn.Linear(32, n_classes)
        self.rank = rank
        self.d_last = 32
        self.task_adapters = nn.ModuleDict()
        self.current_task = 0

    def add_task_adapter(self, task_id):
        adapter = nn.ModuleDict({
            'down': nn.Linear(self.d_last, self.rank, bias=False),
            'up': nn.Linear(self.rank, self.d_last, bias=False)
        })
        nn.init.zeros_(adapter['up'].weight)
        self.task_adapters[str(task_id)] = adapter
        self.current_task = task_id

    def forward(self, x, task_id=None):
        h = self.base(x)
        tid = str(task_id if task_id is not None else self.current_task)
        if tid in self.task_adapters:
            adapter = self.task_adapters[tid]
            h = h + adapter['up'](adapter['down'](h))
        return self.head(h)

class ContinualLearningBenchmark:
    def __init__(self, d=64, n_tasks=3, n_classes_per_task=3):
        self.d = d
        self.n_tasks = n_tasks
        self.n_classes_per_task = n_classes_per_task
        self.tasks = []
        for t in range(n_tasks):
            x = torch.randn(32, d) + t * 0.5
            y = torch.randint(t * n_classes_per_task, (t+1) * n_classes_per_task, (32,))
            self.tasks.append((x, y))

    def evaluate(self, model, up_to_task=None):
        n = up_to_task or self.n_tasks
        accs = []
        for t in range(n):
            x, y = self.tasks[t]
            with torch.no_grad():
                pred = model(x, task_id=t).argmax(1)
                acc = (pred == y).float().mean().item()
            accs.append(acc)
        avg_acc = sum(accs) / len(accs)
        forgetting = max(accs[:-1]) - accs[-1] if len(accs) > 1 else 0
        return {'accs': accs, 'avg_acc': avg_acc, 'forgetting': forgetting}

model_lora = LoRAContinualLearning(d=64, n_classes=9, rank=4)
benchmark = ContinualLearningBenchmark(d=64, n_tasks=3, n_classes_per_task=3)

print('=== LoRA Continual Learning ===')
for task_id in range(3):
    model_lora.add_task_adapter(task_id)
    x, y = benchmark.tasks[task_id]
    optimizer = torch.optim.Adam(model_lora.parameters(), lr=0.01)
    for step in range(30):
        out = model_lora(x, task_id=task_id)
        loss = F.cross_entropy(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    result = benchmark.evaluate(model_lora, up_to_task=task_id+1)
    print(f'After Task {task_id+1}: accs={[f"{a:.2%}" for a in result["accs"]]}, avg={result["avg_acc"]:.2%}')

print(f'\nKey: Each task gets its own LoRA adapter, preventing interference.')
print(f'Base model is frozen, only adapters are trained per task.')
print(f'Inference requires task identity to select the correct adapter.')

## 5. 持续学习总结

| 方法 | 策略 | 优点 | 缺点 | 适用场景 |
|------|------|------|------|----------|
| EWC | 正则化 | 无额外存储 | Fisher计算贵 | 少任务 |
| SI | 正则化 | 在线计算 | 近似不精确 | 中等任务 |
| 经验回放 | 回放 | 简单有效 | 隐私/存储 | 有存储空间 |
| GEM | 梯度投影 | 最优约束 | 计算贵 | 高精度需求 |
| 渐进网络 | 架构 | 无遗忘 | 参数增长 | 任务少 |
| PackNet | 架构 | 固定参数 | 剪枝损失 | 容量充足 |
| LoRA适配器 | 架构 | 无遗忘 | 需任务ID | LLM |

**前沿方向**：
- **O-LoRA**：正交子空间持续学习
- **InfLoRA**：基于扰动分析的LoRA
- **EASE**：扩展自适应子空间编码
- **L2P**：基于提示的持续学习
- **DualPrompt**：双提示分解